# alarm_replication_monthly

Replicate the **prospective rolling-burden alarm** from the manuscript at **monthly granularity** (collapsed 30-day blocks), using the Tabla maestra as gold reference.

**Paper definition (Patients and methods):**
- `pUF(m)` = sum of LDA probabilities for unfavourable DCABPs (topics 3+4+5) in month *m*
- `AUC_W(m)` = rolling sum of `pUF` over the last *W* months
- Alarm at month *m* if `AUC_W(m) ≥ θ`
- Target = progression within the next *H* months
- Operating point reported: **W=3**, **H=4**, **θ=1.61** (mean pUF ≥ 0.54)

Next step (revision branch): repeat with daily / weekly / biweekly granularities.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import auc, roc_curve

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "scripts/03_analysis/alarm"))
sys.path.insert(0, str(ROOT / "scripts/03_analysis/lda"))

from rolling_burden_alarm import (
    build_decision_points,
    default_master_table_path,
    extract_monthly_puf_from_master,
    metrics_at_threshold,
    sweep_thresholds,
    youden_optimal_threshold,
)

MASTER_PATH = default_master_table_path(ROOT)
OUT_TABLES = ROOT / "results/alarm/tables"
OUT_FIGS = ROOT / "results/alarm/figures"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGS.mkdir(parents=True, exist_ok=True)

W, H = 3, 4
PAPER_THRESHOLD = 1.61

## 1 · Load Tabla maestra and build monthly pUF

In [ ]:
master = pd.read_excel(MASTER_PATH)
print(f"Patients: {len(master)} | Events: {int(master['Evento PD'].sum())}")

monthly_puf = extract_monthly_puf_from_master(master)
monthly_puf.head()

## 2 · Decision-point dataset (W months load, H months horizon)

In [ ]:
decision_points = build_decision_points(monthly_puf, W=W, H=H)
print(
    f"Decision points: {len(decision_points)} | "
    f"Positive targets: {int(decision_points['target'].sum())} | "
    f"Patients: {decision_points['id'].nunique()}"
)
decision_points.head()

## 3 · Metrics at manuscript threshold and Youden optimum

In [ ]:
paper_metrics = metrics_at_threshold(decision_points, PAPER_THRESHOLD)
youden_metrics = youden_optimal_threshold(decision_points)

summary = pd.DataFrame([
    {
        "rule": f"paper θ={PAPER_THRESHOLD}",
        "TP": paper_metrics.tp,
        "FN": paper_metrics.fn,
        "FP": paper_metrics.fp,
        "TN": paper_metrics.tn,
        "sensitivity": paper_metrics.sensitivity,
        "specificity": paper_metrics.specificity,
        "PPV": paper_metrics.ppv,
        "NPV": paper_metrics.npv,
    },
    {
        "rule": f"Youden θ={youden_metrics.threshold:.3f}",
        "TP": youden_metrics.tp,
        "FN": youden_metrics.fn,
        "FP": youden_metrics.fp,
        "TN": youden_metrics.tn,
        "sensitivity": youden_metrics.sensitivity,
        "specificity": youden_metrics.specificity,
        "PPV": youden_metrics.ppv,
        "NPV": youden_metrics.npv,
    },
])
summary

## 4 · ROC curve

In [ ]:
fpr, tpr, _ = roc_curve(decision_points["target"], decision_points["AUC_W"])
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color="darkorange", lw=2, label=f"AUC = {roc_auc:.2f}")
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.scatter(
    [1 - paper_metrics.specificity],
    [paper_metrics.sensitivity],
    color="red",
    zorder=5,
    label=f"Paper θ={PAPER_THRESHOLD}",
)
ax.set_xlabel("FPR (1 − specificity)")
ax.set_ylabel("TPR (sensitivity)")
ax.set_title(f"Rolling-burden alarm (W={W}, H={H} months)")
ax.legend(loc="lower right")
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUT_FIGS / f"roc_W{W}_H{H}.png", dpi=200)
fig.savefig(OUT_FIGS / f"roc_W{W}_H{H}.svg")
plt.show()

## 5 · Export tables

In [ ]:
decision_points.to_csv(OUT_TABLES / f"decision_points_W{W}_H{H}.csv", index=False)
sweep_thresholds(decision_points).to_csv(OUT_TABLES / f"threshold_sweep_W{W}_H{H}.csv", index=False)
summary.to_csv(OUT_TABLES / f"alarm_metrics_W{W}_H{H}.csv", index=False)
print("Saved to", OUT_TABLES)